# 04 グラフ出力結果の比較

01（または03）で作ったExcelを複数選び、各項目の「加熱シリンダ温度ごとのグラフ（全厚み統合）」と、その「_エラーバー付き」を**PDFに並べて**比較します。
「外れ値あり」（`グラフ出力結果_{元のファイル名}.xlsx`）と「外れ値なし描画」（`グラフ出力結果_外れ値なし描画_{元のファイル名}.xlsx`）の**2つのPDF**を一度に作ります。

## 使い方
1. セル1 → セル2 → セル3 の順に実行します。
2. セル2を実行するとファイル選択ダイアログが出るので、比較したい `グラフ出力結果_～.xlsx` を選びます。
   - 「外れ値あり」と「外れ値なし描画」のどちらを選んでもかまいません。同じフォルダにあるもう片方のExcelも、自動で読み込みます。
   - 同じフォルダの中のファイルなら、まとめて複数選べます。
   - ダイアログが見当たらないときは、VS Code の後ろやタスクバーを確認してください。
3. 「別のフォルダのファイルも追加しますか？」と聞かれます。
   - 「はい」: ダイアログがもう一度開くので、別のフォルダのファイルを追加します。
   - 「いいえ」: 選択を終わります。
4. セル2の実行結果で「読み込み」の一覧を確認します。「スキップ」や「なし」と表示されたExcelは比較に入りません。
5. セル3を実行すると、PDFの保存先を聞かれます。保存する場所と名前を決めてください。

## 出力（PDF 2つ）
| PDF | 中身 |
|---|---|
| 選んだ名前（初期は `グラフ比較結果.pdf`） | 外れ値ありのExcelの比較（外れ値は×で表示） |
| 選んだ名前の後ろに `_外れ値なし描画` を付けた名前（`グラフ比較結果_外れ値なし描画.pdf`） | 外れ値なし描画のExcelの比較（外れ値を除いて計算した平均・標準偏差。×は描きません） |

- どちらのPDFも、**1項目につき2ページ**で、全部で12ページです。
  - 項目は、最高値、最高値の時間、0.7秒前の値、0.7秒前の時間、立ち上がり値、立ち上がり時間の6つです。
  - 項目ごとに「全厚み統合」→「全厚み統合_エラーバー付き」の順に並びます。
- 各ページには、**行 = 選んだファイル、列 = 加熱シリンダ温度** でグラフを並べます。ファイルは、置き場所（フォルダ名）の順に並びます。
- 各グラフの中身は、01のExcelにある「加熱シリンダ温度ごとのグラフ（全厚み統合）」「…_エラーバー付き」と同じです。
  - 厚みごとの平均値の線と、外れ値の×（×と同じ色でショット番号の3桁を表示）を描きます。
  - エラーバー付きのページでは、平均±標準偏差のエラーバーも描きます。
- **縦軸・横軸の範囲は、そのページのすべてのグラフで同じ**です。線の高さや位置をそのまま比べられます。
- 線の色とマーカーは厚みごとに決まっています。凡例はページの右上に1つだけあります。
- そのファイルにない温度のマスには「データなし」と表示します。

## 注意点
- **01と同じフォルダに置いてください。** グラフの色・マーカー・フォントなどは、`01_最高温度関連グラフ作成.ipynb` のセル1を読み込んで使います。
- **使えるのは、数値のシート（`グラフ用統計値`・`外れ値`）が入ったExcelだけです。** 次のExcelは「スキップ」になります。
  - 01に数値のシートを追加する前に作ったExcel → 03（または01）で作り直してください。
  - 02で作ったExcel（`グラフ出力結果_10shot_～.xlsx`）
- 外れ値なし描画のExcelがないとき（01に外れ値なし描画を追加する前に作ったExcelなど）は「なし」と表示され、外れ値ありのPDFだけに入ります。両方のPDFに入れたいときは、03（または01）で作り直してください。
- 選んだExcelは読み込むだけで、変更しません。
- **保存先と同じ名前のPDFを開いたまま実行しないでください。** 上書きできずにエラーになります。
- 詳しい説明は、プロジェクトフォルダ直下の `README.md` を見てください。

In [ ]:
# =====================================================================================
# セル1: ライブラリの準備と設定
# =====================================================================================
import os
import io
import json
import contextlib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
import tkinter as tk
from tkinter import filedialog, messagebox

# グラフの色・マーカー・フォント・外れ値の描き方・数値シート名は、01のセル1を読み込んで共通にする
# （このノートブックと同じフォルダに置いておく）
SOURCE_NOTEBOOK = '01_最高温度関連グラフ作成.ipynb'

# 比較するExcelのファイル名のパターンと接頭辞（グラフのタイトルでは接頭辞を省く）
EXCEL_PATTERN = 'グラフ出力結果_*.xlsx'
EXCEL_PREFIX = 'グラフ出力結果_'

# 列名（01と同じ）
x_col = '射出速度[mm/s]'
thick_col = 'キャビティ厚み[mm]'
temp_col = '加熱シリンダ温度[℃]'

# グローバル変数の初期化
excel_paths = []
compare_data = {}

source_notebook_path = os.path.join(os.getcwd(), SOURCE_NOTEBOOK)
if not os.path.exists(source_notebook_path):
    print(f"エラー: {SOURCE_NOTEBOOK} が見つかりません。このノートブックと同じフォルダに置いてください。\n探した場所: {source_notebook_path}")
else:
    # 01の1つ目のコードセルを実行する（01のセル1の完了メッセージは表示しない）
    with open(source_notebook_path, encoding='utf-8') as f:
        source_cell = next(''.join(cell['source']) for cell in json.load(f)['cells'] if cell['cell_type'] == 'code')
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(source_cell, f"{SOURCE_NOTEBOOK} セル1", 'exec'))

    # 比較するExcelの種類（「グラフ出力結果_」の後ろに付く文字, 種類の名前）。01が1つのCSVから両方を作る
    EXCEL_VARIANTS = [('', '外れ値あり'), (CLEAN_PREFIX, '外れ値なし描画')]

    print("セル1の実行が完了しました。ライブラリと設定（01のセル1）が読み込まれました。")

In [ ]:
# =====================================================================================
# セル2: 比較するExcelの選択と読み込み
# =====================================================================================
if 'STATS_SHEET' not in globals():
    print("エラー: 01の設定が読み込まれていません。セル1を先に実行してください。")
else:
    print("--- ステップ1: 比較するExcelの選択 ---")
    root = tk.Tk()
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True) # ダイアログを最前面に

    # 比べるExcelは別々のフォルダにあるため、「追加しますか？」で「いいえ」を選ぶまでダイアログを繰り返す
    excel_paths = []
    initial_dir = None
    while True:
        selected = filedialog.askopenfilenames(
            title=f"比較したい「{EXCEL_PATTERN}」を選択してください（複数可・選択済み {len(excel_paths)}件）",
            filetypes=[("グラフ出力結果のExcel", EXCEL_PATTERN), ("Excelファイル", "*.xlsx")],
            initialdir=initial_dir,
        )
        if not selected:
            break
        for path in selected:
            path = os.path.normpath(path)
            if os.path.basename(path).startswith('~$'):
                print(f"  除外（Excelが作る一時ファイル）: {path}")
            elif path in excel_paths:
                print(f"  除外（選択済み）: {path}")
            else:
                excel_paths.append(path)
        print(f"選択済み: {len(excel_paths)}件")
        if not messagebox.askyesno("ファイルの追加", f"選択済み: {len(excel_paths)}件\n\n別のフォルダのファイルも追加しますか？"):
            break
        initial_dir = os.path.dirname(os.path.dirname(selected[-1]))  # 次は1つ上のフォルダから選ぶ

    # 種類ごとに {(フォルダ, 元ファイル名): 読み込んだデータ} を入れる
    compare_data = {name_prefix: {} for name_prefix, _ in EXCEL_VARIANTS}
    if not excel_paths:
        print("ファイルが選択されませんでした。処理を中断します。")
    else:
        # どちらの種類のExcelを選んでも、同じ元ファイルのExcelは1組として扱う
        targets = []
        for path in sorted(excel_paths):
            name = os.path.splitext(os.path.basename(path))[0].removeprefix(EXCEL_PREFIX).removeprefix(CLEAN_PREFIX)
            if (os.path.dirname(path), name) not in targets:
                targets.append((os.path.dirname(path), name))

        print(f"--- ステップ2: 数値シートの読み込み（{len(targets)}組。同じフォルダにある外れ値あり・外れ値なし描画の両方を読み込みます） ---")
        for folder, name in targets:
            label = f"{os.path.basename(folder)} / {name}"
            for name_prefix, variant_name in EXCEL_VARIANTS:
                path = os.path.join(folder, f"{EXCEL_PREFIX}{name_prefix}{name}.xlsx")
                if not os.path.exists(path):
                    print(f"  なし（{variant_name}のExcelがありません。01修正前のExcelのときは、03か01で作り直してください）: {path}")
                    continue
                try:
                    with pd.ExcelFile(path) as xls:
                        if STATS_SHEET not in xls.sheet_names or OUTLIER_SHEET not in xls.sheet_names:
                            print(f"  スキップ（比較用の数値シートがありません。01修正前のExcelか、02のExcelです。03か01で作り直してください）: {path}")
                            continue
                        df_stats = xls.parse(STATS_SHEET)
                        df_outliers = xls.parse(OUTLIER_SHEET)
                except Exception as e:
                    print(f"  スキップ（読み込めません: {type(e).__name__}: {e}）: {path}")
                    continue
                compare_data[name_prefix][(folder, name)] = {'label': label, 'stats': df_stats, 'outliers': df_outliers}
                print(f"  読み込み（{variant_name}）: {label}（{len(df_stats)}条件、×で描く外れ値 {len(df_outliers)}点）")

        all_data = [data for files_data in compare_data.values() for data in files_data.values()]
        if all_data:
            # 項目は「_平均値」で終わる列から取る（01の項目の順）。温度と厚みは全ファイル分を合わせる
            y_cols = []
            for data in all_data:
                y_cols += [c.removesuffix('_平均値') for c in data['stats'].columns if c.endswith('_平均値') and c.removesuffix('_平均値') not in y_cols]
            all_temps = sorted(set().union(*(data['stats'][temp_col].tolist() for data in all_data)))
            all_thicks = sorted(set().union(*(data['stats'][thick_col].tolist() for data in all_data)))

            counts = " / ".join(f"{variant_name} {len(compare_data[name_prefix])}件" for name_prefix, variant_name in EXCEL_VARIANTS)
            print(f"\n比較するファイル: {counts} / 項目: {len(y_cols)}個 / 温度: {all_temps} / 厚み: {all_thicks}")
            print("読み込みが完了しました。次のセルへ進んでください。")
        else:
            print("\n比較できるファイルがありませんでした。")

In [ ]:
# =====================================================================================
# セル3: 比較PDFの作成（外れ値あり・外れ値なし描画の2つ。1ページ = 1項目の1種類、行 = ファイル、列 = 加熱シリンダ温度）
# =====================================================================================
# 1項目につき、01の「加熱シリンダ温度ごとのグラフ（全厚み統合）」と、その「_エラーバー付き」の2ページを作る
GRAPH_TYPES = [('全厚み統合', False), ('全厚み統合_エラーバー付き', True)]

def save_compare_pdf(pdf_path, files_data, title_tag):
    """files_data（ファイルごとの数値シート）の比較グラフをPDFに保存し、ページ数を返す"""
    n_rows, n_cols = len(files_data), len(all_temps)
    n_pages = 0
    with PdfPages(pdf_path) as pdf:
        for y_col in y_cols:
            for graph_name, with_errorbar in GRAPH_TYPES:
                # ページ内のすべてのグラフで縦軸・横軸をそろえる
                fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 3.8 * n_rows + 1.5),
                                         sharex=True, sharey=True, squeeze=False, layout='constrained')
                note = "エラーバーは平均±標準偏差。" if with_errorbar else ""
                fig.suptitle(f"[温度別 統合グラフ（{graph_name}）の比較{title_tag}] {y_col}\n（{note}縦軸・横軸の範囲は、このページのすべてのグラフで共通です）")
                has_outliers = False

                for r, data in enumerate(files_data.values()):
                    df_stats = data['stats']
                    y_outliers = data['outliers'][data['outliers']['項目'] == y_col]

                    for c, temp in enumerate(all_temps):
                        ax = axes[r, c]
                        ax.set_title(f"{data['label']}\n加熱シリンダ温度: {temp}℃", fontsize=9)
                        ax.grid(True)
                        ax.tick_params(labelbottom=True, labelleft=True)  # 目盛りの数値はすべてのグラフに表示する

                        df_temp = df_stats[df_stats[temp_col] == temp]
                        if df_temp.empty or f"{y_col}_平均値" not in df_temp.columns:
                            ax.text(0.5, 0.5, "データなし", transform=ax.transAxes, ha='center', va='center', color='gray', fontsize=14)
                            continue

                        # 01の「加熱シリンダ温度ごとのグラフ（全厚み統合）」と同じ描き方（外れ値は×と同じ色のショット番号）
                        for j, thick in enumerate(all_thicks):
                            df_plot = df_temp[df_temp[thick_col] == thick].sort_values(x_col)
                            if df_plot.empty:
                                continue
                            color = THICK_COLORS.get(thick, COLORS[j % len(COLORS)])
                            marker = THICK_MARKERS.get(thick, 's')
                            pts = y_outliers[(y_outliers[thick_col] == thick) & (y_outliers[temp_col] == temp)]

                            x = df_plot[x_col]
                            y_mean = df_plot[f"{y_col}_平均値"]
                            if with_errorbar:
                                # 数値シートの「標準偏差上・下」（平均±標準偏差）までをエラーバーにする
                                yerr = [y_mean - df_plot[f"{y_col}_標準偏差下"], df_plot[f"{y_col}_標準偏差上"] - y_mean]
                                ax.errorbar(x, y_mean, yerr=yerr, marker=marker, linestyle='-', color=color, capsize=4)
                            else:
                                ax.plot(x, y_mean, marker=marker, linestyle='-', color=color)
                            has_outliers |= plot_outliers(ax, pts, x_col, color, annotate=True)

                # 軸の名前は、下の段と左の列だけに付ける
                for ax in axes[-1, :]:
                    ax.set_xlabel(x_col)
                for ax in axes[:, 0]:
                    ax.set_ylabel(y_col)

                # 凡例はページに1つ（厚みごとの色・マーカーは全グラフで共通）
                handles = [Line2D([], [], marker=THICK_MARKERS.get(thick, 's'), color=THICK_COLORS.get(thick, COLORS[j % len(COLORS)]), label=f"{thick}mm")
                           for j, thick in enumerate(all_thicks)]
                if has_outliers:
                    handles.append(OUTLIER_HANDLE)
                fig.legend(handles=handles, title="キャビティ厚み", loc='outside right upper', frameon=False)

                pdf.savefig(fig)
                plt.close(fig)
                n_pages += 1
                print(f"  ページ作成: {y_col}（{graph_name}）")
    return n_pages


if 'compare_data' not in globals() or not any(compare_data.values()):
    print("エラー: 比較するデータがありません。セル2を先に実行してください。")
else:
    print("--- ステップ3: PDFの保存先の選択（外れ値なし描画のPDFは、名前の後ろに「_外れ値なし描画」が付きます） ---")
    pdf_path = filedialog.asksaveasfilename(
        title="比較結果のPDFの保存先を選択してください",
        initialdir=os.path.commonpath([folder for files_data in compare_data.values() for folder, _ in files_data]),
        initialfile='グラフ比較結果.pdf',
        defaultextension='.pdf',
        filetypes=[("PDFファイル", "*.pdf")],
    )

    if not pdf_path:
        print("保存先が選択されませんでした。処理を中断します。")
    else:
        saved = []
        for name_prefix, variant_name in EXCEL_VARIANTS:
            files_data = compare_data[name_prefix]
            # 外れ値ありは選んだ名前のまま、外れ値なし描画は名前の後ろに「_外れ値なし描画」を付ける
            variant_pdf_path = f"{os.path.splitext(pdf_path)[0]}_{variant_name}.pdf" if name_prefix else pdf_path
            print(f"\n--- ステップ4: 比較グラフの作成（{variant_name}） ---")
            if not files_data:
                print(f"{variant_name}のExcelが1つもないため、このPDFは作りません。")
                continue
            try:
                n_pages = save_compare_pdf(variant_pdf_path, files_data, f"・{variant_name}" if name_prefix else "")
            except PermissionError:
                plt.close('all')  # 作りかけの図を画面に出さない
                print(f"エラー: PDFを保存できませんでした。同じ名前のPDFを開いている場合は閉じてから、もう一度実行してください。\n{variant_pdf_path}")
            else:
                saved.append(f"  {variant_name}: {variant_pdf_path}（{n_pages}ページ、{len(files_data)}ファイル × {len(all_temps)}温度）")

        if saved:
            print("\n比較PDFを保存しました。\n" + "\n".join(saved))